# Iris Task 1 - Classification Project


In [ ]:
# --- Import libraries ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score

#### Load data from iris.data file 

In [13]:
data_path = "/Users/artaosmani/TTT4275-classification-project/Data/Iris_TTT4275/iris.data"
col_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
df = pd.read_csv(data_path, header=None, names=col_names)

# connect class names to numbers
label_map = {
    'Iris-setosa': 0,
    'Iris-versicolor': 1,
    'Iris-virginica': 2
}
df['label'] = df['class'].map(label_map)
print("First 5 rows with new coulumn named label:")
print(df.head())

First 5 rows with new coulumn named label:
   sepal_length  sepal_width  petal_length  petal_width        class  label
0           5.1          3.5           1.4          0.2  Iris-setosa      0
1           4.9          3.0           1.4          0.2  Iris-setosa      0
2           4.7          3.2           1.3          0.2  Iris-setosa      0
3           4.6          3.1           1.5          0.2  Iris-setosa      0
4           5.0          3.6           1.4          0.2  Iris-setosa      0



##### a) Use first 30 samples for training, last 20 for testing 


In [20]:
df = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'label']]

# Function to split the data -> spliting iris.data  in 3 instead of opening 3 separate files
def split_train_test(df, n_train=30, n_test=20):
    train_df = pd.concat([df[df['label'] == i].iloc[:n_train] for i in range(3)])
    test_df = pd.concat([df[df['label'] == i].iloc[n_train:n_train+n_test] for i in range(3)])
    
    X_train = train_df.iloc[:, :-1].values #making a matrix with all rows and all columns exept label
    y_train = train_df['label'].values
    X_test = test_df.iloc[:, :-1].values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = split_train_test(df)
print("X_train:", X_train.shape, "| Eksempel:", X_train[0])

X_train: (90, 4) | Eksempel: [5.1 3.5 1.4 0.2]


#### self defined linear clasifier (MSE + Gradient decent)

Train linear classifier using gradient descent on MSE loss.

X: input shape (n_samples, n_features)

y: class labels (n_samples,)

Returns: weight matrix W of shape (num_classes, n_features+1)
    

Sigmoid function from chapter 3.2: 

$$
g_{ik} = \text{sigmoid}(z_{ik}) = \frac{1}{1 + e^{-z_{ik}}}
$$



In [24]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

One hot encode:

In [ ]:
def one_hot_encode(y, num_classes):
    return np.eye(num_classes)[y]

Computing MSE from equation (19):

$$
\text{MSE} = \frac{1}{2} \sum_{k=1}^{N} (g_k - t_k)^T (g_k - t_k)
$$


In [25]:
def mse(g, t):
    return 0.5 * np.mean(np.sum((g - t) ** 2, axis=1)) 

Model training: Trains a linear classifier using MSE and sigmoid, according to Equations (19)–(23).


| Code                                 | Matches Compendium                     |
|--------------------------------------|----------------------------------------|
| `Z = X_bias @ W.T`                   | \( z_k = W x_k \), Eq. 20              |
| `G = sigmoid(Z)`                     | \( g_k = \sigma(z_k) \), Eq. 20        |
| `loss = mse(G, T)`                   | MSE from Eq. 19                        |
| `dG = (G - T) * G * (1 - G)`         | Gradient components from Eq. 22        |
| `gradient = dG.T @ X_bias / S`       | Full \( \nabla_W \text{MSE} \), Eq. 22 |
| `W = W - alpha * gradient`           | Update rule from Eq. 23                |
| `one_hot_encode(y, C)`              | Vectorial target format (needed for Eq. 19) |


In [26]:
def train_linear_classifier(X, y, alpha=0.1, iterations=1000, threshold=1e-5):
    S, F = X.shape # S = numbers samples, F= number of features
    C = 3 #Numbers of classes
    
    ones = np.ones((S, 1))
    X_bias = np.hstack((X, ones))  
    F += 1                         

    T = one_hot_encode(y, C)       

    W = np.random.randn(C, F) * 0.01  

   
    for i in range(iterations):  # Gradient descent loop
        Z = X_bias @ W.T             
        G = sigmoid(Z)               
        loss = mse(G, T)            

        if loss < threshold:
            print("Converged at iteration", i, "with loss =", loss)
            break

        
        dG = (G - T) * G * (1 - G) #computing gradient      
        gradient = dG.T @ X_bias / S         
        W = W - alpha * gradient

    return W


def predict(X, W):
    X_bias = np.hstack((X, np.ones((X.shape[0], 1))))
    G = sigmoid(X_bias @ W.T)
    return np.argmax(G, axis=1)